In [1]:
import os
import sys
import logging
import matplotlib.pyplot as plt
import seaborn as sns

# Configurar logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

from src.data.load_data import cargar_datos

# Agregar src/ al path si no está
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

csv_path = "../data/raw/vehiculos.csv"

# Cargar datos
df = cargar_datos(csv_path)

2025-07-02 08:00:23,966 - INFO - Datos cargados correctamente desde: ../data/raw/vehiculos.csv


In [2]:
# 1. Eliminar variables específicas
variables_a_eliminar = [
    'fabricante', 'modelo', 'transmision',
    'traccion', 'clase', 'combustible', 'consumo'
]

from src.preprocessing.preparacion import eliminar_variables

df_filtrado = eliminar_variables(df, variables_a_eliminar)

2025-07-02 08:00:24,725 - INFO - Variables eliminadas: ['fabricante', 'modelo', 'transmision', 'traccion', 'clase', 'combustible', 'consumo']


In [3]:
# 2. Separar variables numéricas y categóricas
from src.preprocessing.preparacion import separar_variables

df_num, df_cat = separar_variables(df_filtrado)

2025-07-02 08:00:24,773 - INFO - Variables numéricas: ['year', 'desplazamiento', 'cilindros', 'co2', 'consumo_litros_milla']
2025-07-02 08:00:24,775 - INFO - Variables categóricas: ['clase_tipo', 'traccion_tipo', 'transmision_tipo', 'combustible_tipo', 'tamano_motor_tipo', 'consumo_tipo', 'co2_tipo']


In [4]:
# 3. Imputación basada en porcentaje de nulos
from src.preprocessing.preparacion import imputar_nulos

ruta_salida = "../outputs/02_preparacion"
df_imputado = imputar_nulos(df_num, k=3, umbrales={"bajo": 0.03, "medio": 0.15}, save_dir=ruta_salida)

2025-07-02 08:00:24,839 - INFO - desplazamiento: Se eliminaron 2 filas con nulos (<3%)
2025-07-02 08:00:24,844 - INFO - cilindros: Se eliminaron 3 filas con nulos (<3%)


In [5]:
from src.preprocessing.preparacion import detectar_outliers_univariado

# 4. Detección de outliers univariados
ruta_outliers_uni = "../outputs/02_preparacion"

df_outliers_uni = detectar_outliers_univariado(df_imputado, ruta_salida=ruta_outliers_uni)

2025-07-02 08:00:24,910 - INFO - year: 0 outliers univariados
2025-07-02 08:00:24,918 - INFO - desplazamiento: 43 outliers univariados
2025-07-02 08:00:24,924 - INFO - cilindros: 735 outliers univariados
2025-07-02 08:00:24,931 - INFO - co2: 232 outliers univariados
2025-07-02 08:00:24,938 - INFO - consumo_litros_milla: 1392 outliers univariados


In [6]:
from src.preprocessing.preparacion import detectar_outliers_mahalanobis

# 5. Detección de outliers multivariados
ruta_outliers_multi = "../outputs/02_preparacion"

outliers_maha = detectar_outliers_mahalanobis(
    df_imputado,
    umbral=0.99,
    ruta_salida=ruta_outliers_multi
)

2025-07-02 08:00:25,164 - INFO - Outliers multivariados detectados: 1866


Se observa que hay una distribución de las distancias de Mahalanobis con cola derecha, donde
bajo una distribución chi-cuadrado se tienen 1866 valores atípicos fuera del umbral del 99%.

In [7]:
# Eliminar registros con outliers multivariados
df_sin_outliers = df_imputado.loc[~outliers_maha]

In [8]:
# 6. Escalamiento de variables numéricas
from src.preprocessing.preparacion import escalar_variables

df_escalado = escalar_variables(df_sin_outliers)

2025-07-02 08:00:30,521 - INFO - Escalamiento aplicado a variables numéricas


In [9]:
# 7. Dumificación de variables categóricas
from src.preprocessing.preparacion import dumificar_variables

# Filtrar registros eliminados en dataframe numérico
df_cat_filtrado = df_cat.loc[df_sin_outliers.index]
df_dummies = dumificar_variables(df_cat_filtrado)

2025-07-02 08:00:30,617 - INFO - Dumificación completada


In [10]:
# 8. Unión de dataframes
import pandas as pd
df_modelo = pd.concat([df_escalado, df_dummies], axis=1)
logging.info(f"Data final para modelado: {df_modelo.shape[0]} filas, {df_modelo.shape[1]} columnas")

2025-07-02 08:00:30,672 - INFO - Data final para modelado: 34922 filas, 28 columnas


In [11]:
# Verificar si hay nulos en df_modelo
nulos_modelo = df_modelo.isnull().sum()
nulos_totales = nulos_modelo.sum()

if nulos_totales == 0:
    logging.info("No se encontraron valores nulos en df_modelo.")
else:
    logging.warning(f"Se encontraron {nulos_totales} valores nulos en df_modelo:")
    logging.warning("\n" + str(nulos_modelo[nulos_modelo > 0]))

2025-07-02 08:00:30,708 - INFO - No se encontraron valores nulos en df_modelo.


In [13]:
from pathlib import Path

# Crear carpeta si no existe
Path("../data/processed").mkdir(parents=True, exist_ok=True)

# Exportar a CSV
ruta_csv = "../data/processed/vehiculos_limpios.csv"
df_modelo.to_csv(ruta_csv, index=False)

# Log final
logging.info(f"Data limpia exportada a {ruta_csv}")

2025-07-02 08:01:41,547 - INFO - Data limpia exportada a ../data/processed/vehiculos_limpios.csv
